# Task B Run 20 -- can a multilingual LLM read Kanglish better than MuRIL?

**Question.** Does a 4-bit QLoRA-fine-tuned decoder LLM, with a classification head on the
last token, beat the current best recipe (TAPT MuRIL, one re-initialised layer, R-Drop,
five seeds: the `0.6410` CodaBench submission) on the same held-out rows?

**Why this and not another tweak.** On Task B only representation-level changes have ever
helped: TAPT +2.9 and layer re-initialisation +3.0. Every feature, decode or loss trick
measured as noise or worse. Per-class F1 shows where macro-F1 is lost: Violence 0.35,
Others 0.48, Geo-political 0.59. Violence is carried by romanized verbs
(`saayisbeku`, `encounter maadlebeku`, `odiri`) that MuRIL's wordpieces shatter. An LLM
has seen far more romanized Indic text.

**Data.** `multiclass_train.csv` plus the released labelled validation, deduplicated to
**3,532 rows**. A fixed stratified **15% holdout (530 rows)**, fingerprint `f85f4f049b`,
identical in Run 21. No cross-task label derivation. The test file is never read for
training, TAPT or screening.

| stage | what | time on T4 x2 |
|---|---|---|
| 1a, GPU 0 | **screen**: each candidate LLM in 4-bit scores bits-per-character on unlabelled training comments, one at a time, download deleted after | ~60 min |
| 1b, GPU 1, same time | **baseline**: Run 9's MuRIL recipe trained on the 3,002 training rows, predicting the holdout | ~80 min |
| 2, both GPUs | the two best-screened LLMs, QLoRA seed 42, one per GPU | ~60-100 min |
| 3 | per-class F1, paired bootstrap of LLM - MuRIL, 50/50 ensemble check | 1 min |

**Before running:**
- **Accelerator:** `GPU T4 x2`. **Internet:** on.
- **Gemma only:** accept the license on the Hugging Face model page for
  `google/gemma-4-12B` and `google/gemma-4-E4B`, then add your token as a Kaggle secret
  named `HF_TOKEN` (Add-ons -> Secrets). Without it the Gemma candidates are skipped and
  everything else runs.
- Then **Save Version -> Save & Run All**. Outputs go to `/kaggle/working/b20_outputs/`.

In [ ]:
import glob, json, os, pathlib, shutil, subprocess, sys

REPO = "https://github.com/robinpnalex/Hastika-ICON2026.git"
BRANCH = "task-b"
WORK = "/kaggle/working/hastika"


def git(*args):
    return subprocess.run(["git", "-C", WORK, *args], check=True,
                          capture_output=True, text=True).stdout.strip()


# Never trust an existing clone (the Run 18/21 failures): resolve the remote tip first,
# so "Internet off" fails here with a clear message, then hard-reset to it.
try:
    remote = subprocess.run(["git", "ls-remote", "--exit-code", REPO, f"refs/heads/{BRANCH}"],
                            check=True, capture_output=True, text=True,
                            timeout=60).stdout.split()[0]
except Exception as e:
    raise SystemExit("Cannot reach GitHub. In Kaggle turn on Settings -> Internet (needs a "
                     "phone-verified account), then restart the session.") from e
if os.path.isdir(WORK + "/.git"):
    git("fetch", "-q", "--depth", "1", "origin", BRANCH)
    git("reset", "-q", "--hard", "FETCH_HEAD")
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, "--depth", "1", REPO, WORK], check=True)
head = git("rev-parse", "HEAD")
assert head == remote, f"clone is at {head[:8]} but {BRANCH} is at {remote[:8]}"
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
print("repo at", git("log", "-1", "--oneline"))

# system environment: the repo's MuRIL trainer, exactly as Run 19 installed it
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN secret loaded")
except Exception:
    print("no HF_TOKEN secret: gated models (Gemma) will be skipped, the rest still run")

import numpy as np
import pandas as pd
import torch

assert torch.cuda.is_available(), "select GPU T4 x2"
N_GPU = torch.cuda.device_count()
print("gpus:", [torch.cuda.get_device_name(i) for i in range(N_GPU)])
if N_GPU < 2:
    print("WARNING: one GPU only -- jobs run one after another and take about twice as long")
print(f"free disk on /tmp: {shutil.disk_usage('/tmp').free / 2**30:.0f} GB")

OUT = pathlib.Path("/kaggle/working/b20_outputs")
RUNS = OUT / "runs"
LOGS = OUT / "logs"
for p in (RUNS, LOGS):
    p.mkdir(parents=True, exist_ok=True)
HF_CACHE = "/tmp/hf"          # model downloads stay off the 20 GB /kaggle/working limit
os.environ["HF_HUB_CACHE"] = HF_CACHE

from hastika.common.gpu_queue import run_queue
from hastika.task_b.combined_holdout import (LABELS, TEST_FILE, fingerprint, paired_bootstrap,
                                             report, split)

# The split is committed to the repo, so it cannot drift with a library version on Kaggle.
SPLIT = pathlib.Path(WORK) / "data/derived/task_b_combined_holdout"
df = pd.read_csv(SPLIT / "all.csv")
holdout = pd.read_csv(SPLIT / "holdout.csv")
assert len(df) == 3532 and len(holdout) == 530, (len(df), len(holdout))
HOLDOUT_FP = fingerprint(holdout["id"])
assert HOLDOUT_FP == "f85f4f049b", HOLDOUT_FP      # same rows in every notebook
check_df, _, ho_check = split()
if fingerprint(check_df["id"].iloc[ho_check]) != HOLDOUT_FP:
    print("note: recomputing the split here gives different rows (library versions); "
          "the committed split is used")
y_ho = holdout["Hate Category"].map(LABELS.index).to_numpy()
TEST_CSV = pathlib.Path("/kaggle/working/test_inputs.csv")
pd.read_csv(TEST_FILE)[["id", "Comment"]].to_csv(TEST_CSV, index=False)
print(f"train {len(df) - len(holdout)}, holdout {len(holdout)} ({HOLDOUT_FP}), test {len(pd.read_csv(TEST_CSV))}")

## LLM environment

The LLM stack runs in its own virtual environment, so the MuRIL baseline keeps the
repo's transformers version and both can run at the same time on different GPUs.

In [ ]:
# A separate environment for the LLM stack: Gemma-4 needs transformers >= 5.12, the
# repo's MuRIL trainer is pinned below that. --system-site-packages reuses Kaggle's torch
# and CUDA; --without-pip avoids ensurepip, which Kaggle's Python lacks, and the
# system pip installs into the venv because it runs under the venv's interpreter.
LLM_PY = "/tmp/llmenv/bin/python"
if not os.path.exists(LLM_PY):
    subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", "--without-pip",
                    "/tmp/llmenv"], check=True)
subprocess.run(f'{LLM_PY} -m pip install -q "transformers>=5.12,<6" "peft>=0.17" '
               f'"bitsandbytes>=0.46" "accelerate>=1.0" emoji ftfy sentencepiece protobuf',
               shell=True, check=True)
subprocess.run([LLM_PY, "-c", "import torch, transformers, peft, bitsandbytes; "
                "print('LLM env: torch', torch.__version__, '| transformers', "
                "transformers.__version__, '| peft', peft.__version__, '| bnb', "
                "bitsandbytes.__version__)"], check=True)


def slug(model):
    return model.split("/")[-1].lower()


def download(model):
    """Weights only (safetensors), once, before parallel jobs read the same cache."""
    from huggingface_hub import snapshot_download
    snapshot_download(model, cache_dir=HF_CACHE, token=os.environ.get("HF_TOKEN") or None,
                      allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt",
                                      "tokenizer*", "*.jinja"])
    print(f"downloaded {model}; {shutil.disk_usage('/tmp').free / 2**30:.0f} GB free")


def llm_job(model, name, train, predict, seed, evalf="", extra=""):
    ev = f"--eval {evalf}" if evalf else ""
    return {"name": name, "log": str(LOGS / f"{name}.log"),
            "cmd": (f"{LLM_PY} -u -m hastika.task_b.llm_classifier --model {model} "
                    f"--train {train} {ev} --predict {' '.join(map(str, predict))} "
                    f"--out {RUNS / name} --seed {seed} --cache {HF_CACHE} {extra}")}


def run_with_oom_retry(jobs):
    """One retry at half the micro-batch (same effective batch) for any job that ran
    out of GPU memory. Exit code 3 is fp16 overflow: not retried, reported."""
    codes = run_queue(jobs, N_GPU, raise_on_fail=False)
    retry = []
    for j in jobs:
        if codes.get(j["name"]) and "OutOfMemoryError" in open(j["log"], errors="replace").read():
            print(f"retrying {j['name']} with --bs 4 --grad-accum 4 after OOM")
            retry.append({**j, "cmd": j["cmd"] + " --bs 4 --grad-accum 4 --eval-bs 8"})
    if retry:
        codes.update(run_queue(retry, N_GPU, raise_on_fail=False))
    for n, c in codes.items():
        if c == 3:
            print(f"{n}: fp16 overflow, this model cannot train on a T4")
    return codes

## Stage 1: screen the candidates while the baseline trains

**Candidates.** All fit one T4 in 4-bit: 12B is about 7.5 GB of weights, 8B about 5.5 GB.

| model | why it is here |
|---|---|
| `Qwen/Qwen3-8B-Base` | strong multilingual base, fp16-safe |
| `google/gemma-4-12B` | largest that fits a T4; Gemma's tokenizer is the most Indic-aware |
| `google/gemma-4-E4B` | small Gemma-4, a cheap fallback if the 12B overflows in fp16 |
| `bodhan-ai/indic-translate` | Gemma-4-E4B already fine-tuned to translate **romanized** Indic text |
| `sarvamai/sarvam-1` | 2B, trained on 2T Indic tokens |

**Screen metric.** Bits per character on 600 unlabelled training comments, comparable
across tokenizers. A failure (gated, unsupported, fp16 NaN, OOM) is recorded and skipped.

**Baseline.** Run 9's recipe in a copy of the repo whose inference file holds the holdout
rows. TAPT reads only the 3,002 training comments plus the external Kannada corpus: never
holdout or test text.

In [ ]:
CANDIDATES = ["Qwen/Qwen3-8B-Base", "google/gemma-4-12B", "google/gemma-4-E4B",
              "bodhan-ai/indic-translate", "sarvamai/sarvam-1"]

def muril_workdir(path, infer_csv):
    """A copy of the repo whose `multiclass_validation_inputs.csv` holds the rows to
    predict. train.py always predicts that file, so this scores the holdout (or the test
    set) with zero source changes, and the git clone itself is never modified."""
    shutil.rmtree(path, ignore_errors=True)
    shutil.copytree(WORK, path, ignore=shutil.ignore_patterns(".git", "artifacts"))
    pd.read_csv(infer_csv)[["id", "Comment"]].to_csv(
        f"{path}/data/raw/multiclass_validation_inputs.csv", index=False)
    return path


def muril_job(name, path, train_csv):
    """Run 9's recipe, the 0.6410 CodaBench submission: TAPT on the training text plus
    the external Kannada corpus, one re-initialised layer, R-Drop 0.5, five seeds."""
    tapt = f"artifacts/runs/tapt-{name}"
    cmd = (f"set -e; python -u -m hastika.task_b.tapt --model google/muril-base-cased "
           f"--corpus {train_csv} data/external/offenseval_kn.csv --val-frac 0 "
           f"--min-words 1 --no-dedupe --epochs 8 --out {tapt}; "
           f"python -u -m hastika.task_b.train --tag {name} --model {tapt} "
           f"--train-data {train_csv} --folds 1 --no-dedupe --reinit-layers 1 --rdrop 0.5 "
           f"--aux-weight 0 --seeds 42 43 44 45 46 --epochs 6")
    return {"name": name, "cmd": cmd, "cwd": path, "log": str(LOGS / f"{name}.log"),
            "env": {"PYTHONPATH": f"{path}/src"}}


def muril_probs(path, name):
    p = pathlib.Path(path) / "artifacts/runs" / name / "test_probs.npy"
    return np.load(p) if p.exists() else None


MWORK = muril_workdir("/tmp/muril_holdout", SPLIT / "holdout.csv")
screen = {"name": "screen", "log": str(LOGS / "screen.log"),
          "cmd": (f"{LLM_PY} -u -m hastika.task_b.llm_screen --texts {SPLIT}/train.csv "
                  f"--out {OUT}/screen.json --cache /tmp/hf_screen --models {' '.join(CANDIDATES)}")}
codes = run_queue([screen, muril_job("b20_muril", MWORK, SPLIT / "train.csv")], N_GPU,
                  raise_on_fail=False)
print(open(LOGS / "screen.log").read()[-2500:])
muril_ho = muril_probs(MWORK, "b20_muril")
if muril_ho is not None:
    np.save(OUT / "muril_holdout_probs.npy", muril_ho)
    report(y_ho, muril_ho, "MuRIL Run 9 recipe")
else:
    print("MuRIL baseline failed -- see", LOGS / "b20_muril.log")

## Stage 2: fine-tune the two best-screened models

Lower bits-per-character means the model already reads this text better. That is a
proxy, not a guarantee, so the top **two** are fine-tuned rather than one. Set
`FINETUNE` by hand to override the pick.

Recipe: LoRA r=16 on every attention and MLP projection, lr 1e-4 (head 5e-4), 3 epochs,
effective batch 16, balanced class weights, label smoothing 0.05. The epoch count is
fixed; the per-epoch holdout score in the log is a diagnostic only and selects nothing.

In [ ]:
FINETUNE = None          # e.g. ["Qwen/Qwen3-8B-Base", "google/gemma-4-12B"] to override

rows = json.load(open(OUT / "screen.json"))
ok = sorted((r for r in rows if r.get("status") == "ok"), key=lambda r: r["bpc"])
for r in rows:
    print(f"  {r.get('bpc', float('nan')):.3f}  {r['model']:28s} {r['status'][:90]}")
TOP = FINETUNE or [r["model"] for r in ok][:2]
assert TOP, "no candidate loaded -- check screen.log (HF_TOKEN? Internet?)"
print("fine-tuning:", TOP)
for m in TOP:
    download(m)

jobs = [llm_job(m, f"{slug(m)}_s42", SPLIT / "train.csv", [SPLIT / "holdout.csv"], 42,
                evalf=SPLIT / "holdout.csv") for m in TOP]
codes = run_with_oom_retry(jobs)

## Stage 3: results

**Paired bootstrap.** Both arms are scored on the same resampled holdout rows 5,000 times.
A single 530-row macro-F1 swings by about +-2.5 points, but most of that swing is shared
by both arms, so the paired interval is far tighter. That is what makes the comparison
readable.

The ensemble is a fixed 50/50 average of probabilities. Fitting the weight has failed its
nested check on every run in this repo, so no weight is searched.

In [ ]:
arms = {}
if muril_ho is not None:
    arms["MuRIL (Run 9 recipe)"] = muril_ho
for m in TOP:
    p = RUNS / f"{slug(m)}_s42" / "holdout_probs.npy"
    if p.exists():
        arms[m] = np.load(p)
        hist = json.load(open(RUNS / f"{slug(m)}_s42" / "metrics.json"))["history"]
        print(m, "per-epoch holdout (diagnostic):",
              [round(h.get("eval_macro_f1", 0), 4) for h in hist])
print()
scores = {k: report(y_ho, v, k[:28]) for k, v in arms.items()}
print("\nholdout class counts:", dict(zip(LABELS, np.bincount(y_ho, minlength=6))))
for k, v in arms.items():
    print(f"predicted counts {k[:24]:24s}", dict(zip(LABELS, np.bincount(v.argmax(1), minlength=6))))

llms = [k for k in arms if k in TOP]
result = {"commit": head, "holdout_fingerprint": HOLDOUT_FP, "screen": rows,
          "scores": scores, "winner": None}
if llms:
    best = max(llms, key=lambda k: scores[k])
    result["winner"] = best
    if muril_ho is not None:
        bs = paired_bootstrap(y_ho, arms[best], muril_ho)
        ens = (arms[best] + muril_ho) / 2
        scores["ensemble 50/50"] = report(y_ho, ens, "ensemble 50/50")
        bse = paired_bootstrap(y_ho, ens, arms[best])
        dis = (arms[best].argmax(1) != muril_ho.argmax(1)).mean()
        print(f"\n{best} - MuRIL: {bs['diff']:+.4f}, 95% CI [{bs['ci95'][0]:+.4f}, "
              f"{bs['ci95'][1]:+.4f}], P(LLM better) {bs['p_better']:.2f}")
        print(f"ensemble - {best}: {bse['diff']:+.4f}, 95% CI [{bse['ci95'][0]:+.4f}, "
              f"{bse['ci95'][1]:+.4f}], P(ensemble better) {bse['p_better']:.2f}")
        print(f"LLM and MuRIL disagree on {dis:.1%} of holdout rows")
        np.save(OUT / "ensemble_holdout_probs.npy", ens)
        result.update(llm_vs_muril=bs, ensemble_vs_llm=bse, disagreement=float(dis))
    np.save(OUT / "llm_holdout_probs_s42.npy", arms[best])
json.dump(result, open(OUT / "result.json", "w"), indent=2)
print("\nwinner:", result["winner"], "-> attach this notebook's output to Run 21")

## Reading the result

- **LLM - MuRIL interval entirely above 0**: the LLM is a real improvement. Run 21
  confirms it with two fresh seeds and builds the final submission.
- **Interval straddles 0, P(LLM better) above 0.7**: promising but not proven. Run 21's
  three-seed average decides.
- **LLM clearly below MuRIL**: record it in `docs/EXPERIMENTS.md` and fall back to the
  MuRIL stack (self-distillation, whole-word-masking TAPT).
- **Screen row says `nan (fp16 overflow)`**: that model cannot train on a T4. That is a
  property of the hardware, not evidence about the model.

The winner was picked between two LLMs on this holdout, so its score here is slightly
optimistic. Run 21's fresh seeds 43 and 44 are not subject to that selection.

Record the screen table, every arm's macro-F1 and per-class F1, and both bootstrap
intervals in `docs/EXPERIMENTS.md`.